In [ ]:
import numpy as np
import networkx as nx
from dataclasses import dataclass, field
from mcr.avinfra_persuasion.orders.partial_order import PartialOrder
import taichi as ti
from abc import abstractmethod
from mcr.avinfra_persuasion.routing.mosp import solve_mosp_routes_from_components
from typing import Callable
from scipy.stats import beta
from itertools import count

ti.init()

[Taichi] Starting on arch=arm64


In [33]:
n_side = 10
G = nx.grid_2d_graph(n_side, n_side)

k = 9

s = int(np.sqrt(k))
block_size = n_side // s

partitions = []
for i in range(s):
    for j in range(s):
        nodes = [
            (row, col)
            for row in range(i * block_size, (i + 1) * block_size)
            for col in range(j * block_size, (j + 1) * block_size)
        ]
        partitions.append(nodes)


In [ ]:
metrics = (
    "TRAVEL_TIME", 
    "HAZARD", 
    "COST"
)

@dataclass
class Timer:
    """Singleton Timer"""
    time: int = 0
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            cls._instance = super().__new__(cls)
        return cls._instance 
    
    def step(self):
        self.time += 1 
    
    def reset(self):
        self.time = 0

@dataclass
class Network:
    """Singleton Timer"""
    graph: nx.MultiDiGraph
    prior: "Prior"
    
    _vehicles: dict[tuple[int, int], int]
    
    bpr_alpha: float
    bpr_beta: float
    toll_min: float
    toll_max: float
    
    no_toll_ratio: float = 1.0
    toll_saturation_ratio: float = 2.0
    
    def __post_init__(self):
        if self.toll_min < 0:
            raise ValueError("toll_min must be nonnegative.")

        if self.toll_max < self.toll_min:
            raise ValueError("toll_max must be greater than or equal to toll_min.")

        if self.toll_saturation_ratio <= self.no_toll_ratio:
            raise ValueError(
            "toll_saturation_ratio must be greater than no_toll_ratio."
            )
    
    def register_vehicle(self, edge: tuple[int, int]) -> None:
        self._vehicles[edge] += 1
    
    def deregister_vehicle(self, edge: tuple[int, int]) -> None:
        self._vehicles[edge] -= 1
    
    def _bpr_function(self, t0: float, volume: int, capacity: int) -> float:
        ratio = volume / capacity
        return t0 * (1 + self.bpr_alpha * ratio ** self.bpr_beta)

    def _bpr_toll(self, volume: float, capacity: float) -> float:
        ratio = volume / capacity

        # No toll while traffic is at or below the no-toll threshold
        if ratio <= self.no_toll_ratio:
            return 0.0

        numerator = ratio ** self.bpr_beta - self.no_toll_ratio ** self.bpr_beta
        denominator = (
            self.toll_saturation_ratio ** self.bpr_beta
            - self.no_toll_ratio ** self.bpr_beta
        )

        scaled_congestion = numerator / denominator

        toll = self.toll_min + (
            self.toll_max - self.toll_min
        ) * scaled_congestion

        return float(np.clip(toll, self.toll_min, self.toll_max))
    
    def _get_congestion_tt(self, edge: tuple[int, int]) -> float:
        edge_data = self.graph.get_edge_data(*edge, 0)

        nominal_tt = edge_data["TRAVEL_TIME"]
        nominal_c = edge_data["CAPACITY"]
        current_v = self._vehicles[edge]

        return self._bpr_function(nominal_tt, current_v, nominal_c)
    
    def _get_congestion_hazard(self, edge: tuple[int, int]) -> float:
        nominal_h = self.graph.get_edge_data(*edge, 0)["HAZARD"]
        nominal_c = self.graph.get_edge_data(*edge, 0)["CAPACITY"]
        current_v = self._vehicles[edge]
        return nominal_h + np.floor(current_v/nominal_c)
    
    def _get_toll(self, edge: tuple[int, int]) -> float:
        edge_data = self.graph.get_edge_data(*edge, 0)

        nominal_c = edge_data["CAPACITY"]
        current_v = self._vehicles[edge]

        return self._bpr_toll(volume=current_v, capacity=nominal_c)
        
    def _get_congestion_cost(self, edge: tuple[int, int]) -> float:
        edge_data = self.graph.get_edge_data(*edge, 0)

        nominal_cost = edge_data["COST"]
        toll = self._get_toll(edge)

        return nominal_cost + toll
        
    def get_current_metrics(self, edge: tuple[int, int]):
        return {
            "TRAVEL_TIME": self._get_congestion_tt(edge), 
            "HAZARD": self._get_congestion_hazard(edge), 
            "COST": self._get_congestion_cost(edge)
        }
    
    def get_route_metrics(self, route: list[tuple[int, int]]) -> dict[str, float]:
        edge_metrics = [self.get_current_metrics(edge) for edge in route]

        return {
            "TRAVEL_TIME": sum(m["TRAVEL_TIME"] for m in edge_metrics),
            "HAZARD": max(m["HAZARD"] for m in edge_metrics),
            "COST": sum(m["COST"] for m in edge_metrics),
        }


@dataclass
class NetworkBelief(Network):
    
    @staticmethod
    def from_network(network: Network):
        pass


@dataclass
class CommonNetwork(Network):
    
    _instance = None
    
    def __new__(cls, *args, **kwargs):
        if not cls._instance:
            cls._instance = super().__new__(cls)
        return cls._instance 


@dataclass
class Prior:
    base_network: NetworkBelief


@dataclass
class Player:
    identifier: int = field(default_factory=count().__next__, init=False)
    preference: PartialOrder
    timer: Timer
    prior: Prior
    
    def __post_init__(self):
        if not all(e in self.preference.elements for e in metrics):
            raise ValueError("Make sure the preferences are using the metrics")
        self.timer = Timer()
    
    @abstractmethod
    def step(self):
        pass

@dataclass
class Receiver(Player):
    driver: str
    sender: 'Sender'
    
    origin: tuple[int, int]
    destination: tuple[int, int]
    departure_time: int
    
    routes: dict[str, list[tuple[int, int]]]
    chosen_route: str | None = None
    experienced_metrics: dict = {m: 0 for m in metrics}
    
    def __post_init__(self):
        super().__post_init__()
        self.routes 
    
    def _compute_route_set(self):
        pass
    
    @property
    def type(self):
        return self.driver + str(self.origin) + '_' + str(self.destination)
    
    def _get_world_belief(self):
        pass
    
    @property
    def current_position(self):
        if self.chosen_route is None or self.timer.time <= self.departure_time:
            return self.origin
        
        progress = 0
        for u, v in self.routes[self.choose_route]:
            progress += data['travel_time']
            
    def step(self):
        pass
    
    def update_belief(self, signal: np.ndarray, sender_policy: np.ndarray):
        pass
    
    def choose_route(self):
        pass


@dataclass
class Sender:
    preference: PartialOrder
    signal_policy: np.ndarray
    
    timer: Timer = Timer()
    
    def __post_init__(self):
        pass
    
    def evaluate_world(self):
        pass
    
    def update_policy(self):
        pass


@dataclass
class World:
    
    
    _sender: Sender = field()
    _receivers: list[Receiver] = field()
    graph: nx.MultiDiGraph
    infra_subgraph: nx.MultiDiGraph
    timer: Timer = Timer()
    
    def step(self):
        
        self.timer.step()